# Models

> 1D convolutional neural networks for soil spectra


A spectrum is a 1D signal, with one value per wavenumber. The models here are 1D convolutional neural networks (CNNs) built from fastai's `ConvLayer`. They take batches of shape `(batch, channels, wavenumbers)`, usually with one channel.

In [1]:
#| default_exp models

In [2]:
#| export
from functools import partial
import torch
from torch.nn import Module, AvgPool1d, LeakyReLU, AdaptiveAvgPool1d, Sequential, Dropout, Linear

from fastai.layers import ConvLayer, NormType
from typing import Callable

In [3]:
from fastcore.test import test_eq

In [4]:
#| export
class ConvBlock(Module):
    "Convolution, normalization and activation, then average pooling"
    def __init__(
        self,
        ni, # input channels
        nf, # output channels
        ks=3, # kernel size
        stride=1, # stride
        pool_sz=3, # pooling kernel size
        act_cls=partial(LeakyReLU, negative_slope=0.1), # activation
        norm_type=NormType.Batch, # normalization, e.g. `NormType.Instance`
    ):
        super().__init__()
        self.conv = ConvLayer(ni, nf, ks=ks, stride=stride, bias=False, act_cls=act_cls, norm_type=norm_type, ndim=1)
        self.pool = AvgPool1d(pool_sz)
    def forward(self, x): return self.pool(self.conv(x))

The convolution keeps the number of positions, and the pooling divides it by `pool_sz`. With the default `pool_sz=3`, 64 positions become 21:


In [5]:
x = torch.randn(32, 61, 64)
block = ConvBlock(61, 128)
out = block(x)
test_eq(out.shape, (32, 128, 21))
out.shape

torch.Size([32, 128, 21])

In [6]:
#| export
class FeatureExtractor(Module):
    "Stack of `ConvBlock`s, one per entry of `nfs`, pooled to one vector per spectrum"
    def __init__(
        self,
        in_channel=1, # input channels
        nfs=(16, 32, 64, 128, 256), # output channels of each conv block
        pool_sz=3, # pooling kernel size of each block
        act_cls=partial(LeakyReLU, negative_slope=0.1), # activation
        norm_type=NormType.Batch, # normalization, e.g. `NormType.Instance`
        conv_block_cls=ConvBlock, # conv block class
    ):
        super().__init__()
        self.nf = nfs[-1]
        nis = (in_channel, *nfs[:-1])
        self.layers = Sequential(*[conv_block_cls(ni, nf, pool_sz=pool_sz, act_cls=act_cls, norm_type=norm_type)
            for ni, nf in zip(nis, nfs)])
        self.adaptive_pool = AdaptiveAvgPool1d(1)

    def forward(self, x): return self.adaptive_pool(self.layers(x)).squeeze(-1)


`FeatureExtractor` averages over the positions left after its last block. Each spectrum becomes one vector of `nf` features, whatever its length:


In [7]:
fe = FeatureExtractor(in_channel=61)
out = fe(torch.randn(32, 61, 243))
test_eq(out.shape, (32, fe.nf))
out.shape


torch.Size([32, 256])

In [8]:
#| export
class SpectralCNN(Module):
    "1D CNN with a `FeatureExtractor` backbone and a linear head, for regression or binary classification"
    def __init__(
        self,
        in_channel:int=1, # input channels
        nfs=(16, 32, 64, 128, 256), # output channels of each conv block
        pool_sz:int=3, # pooling kernel size of each block
        act_cls=partial(LeakyReLU, negative_slope=0.1), # activation
        norm_type=NormType.Batch, # normalization, e.g. `NormType.Instance`
        conv_block_cls:Callable=ConvBlock, # conv block class
        is_classifier:bool=False, # if True, apply a sigmoid to output probabilities
        dropout:float=0.4 # dropout rate
    ):
        super().__init__()
        self.backbone = FeatureExtractor(in_channel, nfs, pool_sz=pool_sz, act_cls=act_cls,
            norm_type=norm_type, conv_block_cls=conv_block_cls)
        self.head = Sequential(Dropout(dropout), Linear(self.backbone.nf, 1))
        self.is_classifier = is_classifier

    def forward(self, x):
        x = self.head(self.backbone(x)).squeeze(-1)
        return torch.sigmoid(x) if self.is_classifier else x


`SpectralCNN` returns one value per spectrum. With `is_classifier=True`, a sigmoid turns that value into a probability between 0 and 1:


In [9]:
model_reg = SpectralCNN()
model_cls = SpectralCNN(is_classifier=True)

x = torch.randn(32, 1, 1701)
pred_reg = model_reg(x)
pred_cls = model_cls(x)

test_eq(pred_reg.shape, (32,))
assert 0 <= pred_cls.min() and pred_cls.max() <= 1
pred_reg.shape, pred_cls.min().item(), pred_cls.max().item()


(torch.Size([32]), 0.38777458667755127, 0.6036885380744934)

`MirzaiCNN` builds the architecture from Albinet et al. (2022), [Prediction of exchangeable potassium in soil through mid-infrared spectroscopy and deep learning: From prediction to explainability](https://doi.org/10.1016/j.aiia.2022.10.001), *Artificial Intelligence in Agriculture* 6, 230–241. It is a `SpectralCNN` with five `BatchNorm` blocks and `AvgPool1d(3)`, sized for MIR spectra of about 1,700 wavenumbers.

In [10]:
#| export
def MirzaiCNN(
    in_channel:int=1, # input channels
    is_classifier:bool=False, # if True, model will output probabilities
    dropout:float=0.4 # dropout rate
)->SpectralCNN:
    "`SpectralCNN` with the architecture of Albinet et al. (2022)"
    return SpectralCNN(in_channel, nfs=(16, 32, 64, 128, 256), pool_sz=3, act_cls=partial(LeakyReLU, negative_slope=0.1),
        norm_type=NormType.Batch, is_classifier=is_classifier, dropout=dropout)

The paper's model has 131,857 parameters:

In [11]:
mirzai = MirzaiCNN()
n_params = sum(p.numel() for p in mirzai.parameters())
test_eq(n_params, 131_857)
n_params

131857